# Uji Statistik Bab IV — ExoVirt

Notebook ini menghasilkan **3 gambar bukti** untuk skripsi:

| Gambar | Isi |
|---|---|
| **4.19** | Uji normalitas Shapiro-Wilk (waktu provisioning, H1) |
| **4.20** | Uji beda Mann-Whitney U (waktu provisioning, H1) |
| **4.27** | SUS: Shapiro-Wilk selisih + paired t-test + Wilcoxon (H3) |

**Cara pakai di Google Colab:**
1. Buka [colab.research.google.com](https://colab.research.google.com) → File → Upload notebook → pilih file ini.
2. Menu **Runtime → Run all** (atau Ctrl+F9).
3. Tiap gambar tersimpan sebagai PNG (`gambar_4_19.png`, dst). Sel terakhir mengunduhnya otomatis.
4. Data sudah terisi dari tabel Bab IV, jadi angkanya sama persis. Ganti angka di sel data kalau perlu.


In [ ]:
# Sel 1 — impor (semua paket sudah tersedia di Colab)
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

import scipy
print('scipy', scipy.__version__, '- semua paket siap.')


## Data H1 — Waktu provisioning (detik)

Portal = `t1 + t3` (10 VM), Manual = `t_manual` (10 VM).

In [ ]:
# Sel 2 — data waktu provisioning
portal = [110, 97, 94, 97, 95, 95, 110, 97, 95, 98]   # t1+t3 (Tabel 4.12)
manual = [175, 146, 138, 150, 130, 129, 128, 121, 123, 129]  # t_manual (Tabel 4.11)
print('n portal =', len(portal), '| n manual =', len(manual))


## Gambar 4.19 — Uji Normalitas Shapiro-Wilk (H1)

Memeriksa **bentuk sebaran** tiap kelompok. p < 0,05 = tidak normal (mencong) → pakai uji non-parametrik.

In [ ]:
# Sel 3 — Gambar 4.19
def shapiro_row(nama, x):
    W, p = stats.shapiro(x)
    return {'Kelompok': nama, 'n': len(x), 'W': round(W, 4), 'p': round(p, 4),
            'Kesimpulan': 'Normal' if p > 0.05 else 'Tidak normal'}

tabel_419 = pd.DataFrame([shapiro_row('Portal (t1+t3)', portal),
                          shapiro_row('Manual (t_manual)', manual)])
display(tabel_419)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (nama, data, warna) in zip(
        axes, [('Portal (t1+t3)', portal, '#4C78A8'), ('Manual (t_manual)', manual, '#F58518')]):
    d = np.array(data)
    ax.hist(d, bins=6, density=True, alpha=0.55, edgecolor='white', color=warna)
    xs = np.linspace(d.min() - 8, d.max() + 8, 200)
    ax.plot(xs, stats.norm.pdf(xs, d.mean(), d.std(ddof=1)), 'r-', lw=2, label='Kurva normal')
    W, p = stats.shapiro(d)
    ax.set_title(f'{nama}\nShapiro-Wilk  W = {W:.4f},  p = {p:.4f}')
    ax.set_xlabel('Waktu (detik)'); ax.set_ylabel('Densitas'); ax.legend()
fig.suptitle('Gambar 4.19  Uji Normalitas Shapiro-Wilk - Waktu Provisioning', fontweight='bold')
plt.tight_layout()
fig.savefig('gambar_4_19.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tersimpan: gambar_4_19.png')


## Gambar 4.20 — Uji Mann-Whitney U (H1)

Dua kelompok bebas. U = 0 karena tidak ada irisan: waktu portal terlambat pun lebih cepat dari waktu manual tercepat.

In [ ]:
# Sel 4 — Gambar 4.20
with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # abaikan peringatan ties; hitung p eksak
    U, p = stats.mannwhitneyu(portal, manual, alternative='two-sided', method='exact')

tabel_420 = pd.DataFrame({
    'Ukuran': ['n portal', 'n manual', 'U', 'p (eksak, 2 sisi)', 'Taraf signifikansi', 'Keputusan'],
    'Nilai': [len(portal), len(manual), f'{U:.0f}', f'{p:.3e}', '0,05',
              'H0 ditolak, H1 diterima' if p < 0.05 else 'H0 gagal ditolak']
})
display(tabel_420)

fig, ax = plt.subplots(figsize=(7, 4.6))
bp = ax.boxplot([portal, manual], labels=['Portal', 'Manual'],
                patch_artist=True, widths=0.5, medianprops=dict(color='black'))
for patch, c in zip(bp['boxes'], ['#4C78A8', '#F58518']):
    patch.set_facecolor(c); patch.set_alpha(0.55)
for i, data in enumerate([portal, manual], start=1):
    ax.scatter(np.random.normal(i, 0.05, len(data)), data, color='black', alpha=0.7, zorder=3, s=25)
ax.axhline(max(portal), ls='--', color='#4C78A8', alpha=0.8)
ax.axhline(min(manual), ls='--', color='#F58518', alpha=0.8)
ax.set_ylabel('Waktu provisioning (detik)')
ax.set_title(f'Gambar 4.20  Mann-Whitney U = {U:.0f},  p = {p:.2e}\n'
             f'Tanpa irisan: maks portal ({max(portal)}) < min manual ({min(manual)})',
             fontweight='bold')
plt.tight_layout()
fig.savefig('gambar_4_20.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tersimpan: gambar_4_20.png')


## Data H3 — Skor SUS (berpasangan, 8 responden)

Setiap responden menilai **kedua** sistem, jadi datanya berpasangan.

In [ ]:
# Sel 5 — data SUS
resp        = ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8']
peran       = ['Pengguna biasa'] * 5 + ['Administrator TI'] * 3
sus_portal  = [77.5, 87.5, 85.0, 92.5, 97.5, 100.0, 97.5, 100.0]
sus_proxmox = [17.5, 25.0, 17.5, 15.0, 17.5,  67.5,  72.5,  70.0]
diff = np.array(sus_portal) - np.array(sus_proxmox)

display(pd.DataFrame({'Responden': resp, 'Peran': peran,
                      'Portal': sus_portal, 'Proxmox': sus_proxmox, 'Selisih': diff}))


## Gambar 4.27 — SUS: Normalitas selisih + Uji Beda (H3)

Shapiro-Wilk pada **selisih** (karena berpasangan) → paired t-test (utama) + Wilcoxon (konfirmasi) + Cohen's d.

In [ ]:
# Sel 6 — Gambar 4.27
Wd, pd_diff = stats.shapiro(diff)
tabel_norm = pd.DataFrame([{'Data': 'Selisih SUS (portal - Proxmox)', 'n': len(diff),
                            'W': round(Wd, 4), 'p': round(pd_diff, 4),
                            'Kesimpulan': 'Normalitas tidak ditolak' if pd_diff > 0.05 else 'Tidak normal'}])
display(tabel_norm)

t, pt = stats.ttest_rel(sus_portal, sus_proxmox)
Wsr, pw = stats.wilcoxon(sus_portal, sus_proxmox)
cohen_d = diff.mean() / diff.std(ddof=1)
tabel_beda = pd.DataFrame({
    'Uji': ['Paired sample t-test', "Cohen's d", 'Wilcoxon signed-rank'],
    'Statistik': [f't({len(diff)-1}) = {t:.3f}', f'{cohen_d:.3f}', f'W = {Wsr:.0f}'],
    'p': [f'{pt:.6f}', '-', f'{pw:.4f}'],
    'Keterangan': ['Uji utama', 'Ukuran efek (sangat besar)', 'Uji konfirmasi']
})
display(tabel_beda)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
# kiri: histogram + kurva normal pada selisih
d = diff
axes[0].hist(d, bins=6, density=True, alpha=0.55, edgecolor='white', color='#54A24B')
xs = np.linspace(d.min() - 12, d.max() + 12, 200)
axes[0].plot(xs, stats.norm.pdf(xs, d.mean(), d.std(ddof=1)), 'r-', lw=2, label='Kurva normal')
axes[0].set_title(f'Normalitas selisih SUS\nShapiro-Wilk  W = {Wd:.4f},  p = {pd_diff:.4f}')
axes[0].set_xlabel('Selisih skor (portal - Proxmox)'); axes[0].set_ylabel('Densitas'); axes[0].legend()
# kanan: slope plot berpasangan
for i in range(len(resp)):
    c = '#4C78A8' if peran[i] == 'Pengguna biasa' else '#F58518'
    axes[1].plot([0, 1], [sus_proxmox[i], sus_portal[i]], '-o', color=c, alpha=0.8)
axes[1].axhline(68, ls='--', color='gray', label='Ambang SUS 68')
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['Proxmox VE', 'Portal'])
axes[1].set_ylabel('Skor SUS'); axes[1].set_title('Skor berpasangan per responden'); axes[1].legend()
fig.suptitle(f'Gambar 4.27  SUS - paired t p = {pt:.2e},  Wilcoxon p = {pw:.4f},  Cohen d = {cohen_d:.2f}',
             fontweight='bold')
plt.tight_layout()
fig.savefig('gambar_4_27.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tersimpan: gambar_4_27.png')


## Ringkasan — cocokkan dengan tabel Bab IV

In [ ]:
# Sel 7 — ringkasan semua angka
wpt, ppt = stats.shapiro(portal)
wmt, pmt = stats.shapiro(manual)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    Uu, pUu = stats.mannwhitneyu(portal, manual, alternative='two-sided', method='exact')
wdd, pdd = stats.shapiro(diff)
tt, ptt = stats.ttest_rel(sus_portal, sus_proxmox)
Wss, pss = stats.wilcoxon(sus_portal, sus_proxmox)
cd = diff.mean() / diff.std(ddof=1)

print(f'H1  Shapiro portal  : W={wpt:.4f}  p={ppt:.4f}   (Bab IV: 0,6871 / 0,0006)')
print(f'H1  Shapiro manual  : W={wmt:.4f}  p={pmt:.4f}   (Bab IV: 0,8370 / 0,0410)')
print(f'H1  Mann-Whitney U  : U={Uu:.0f}  p={pUu:.3e}   (Bab IV: 0 / 1,08e-5)')
print(f'H3  Shapiro selisih : W={wdd:.4f}  p={pdd:.4f}   (Bab IV: 0,8767 / 0,1753)')
print(f'H3  Paired t-test   : t={tt:.3f}  p={ptt:.6f}   (Bab IV: 6,982 / 0,000215)')
print(f'H3  Wilcoxon        : W={Wss:.0f}  p={pss:.4f}   (Bab IV: 0 / 0,0078)')
print(f"H3  Cohen's d       : {cd:.3f}                (Bab IV: 2,468)")


## Unduh gambar (jalankan di Colab)

In [ ]:
# Sel 8 — unduh ketiga PNG (khusus Colab)
try:
    from google.colab import files
    for f in ['gambar_4_19.png', 'gambar_4_20.png', 'gambar_4_27.png']:
        files.download(f)
except Exception as e:
    print('Lewati unduh otomatis (bukan di Colab). File tetap tersimpan di folder kerja.')
